# Fashion MNIST — классификация изображений одежды

**Цель:** обучить нейронную сеть распознавать 10 категорий одежды по изображениям размером 28 × 28 пикселей.

В работе используются TensorFlow/Keras и набор данных Fashion MNIST. Код построен по мотивам `example.ipynb`, но оформлен как самостоятельное решение: подготовка данных, создание и обучение модели, оценка качества, графики и примеры предсказаний.

## 1. Импорт библиотек и воспроизводимость

Используем NumPy для работы с массивами, Matplotlib для графиков и Keras для нейронной сети. Фиксируем случайное начальное значение, чтобы результаты запусков были более сопоставимыми.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

keras.utils.set_random_seed(42)
print("TensorFlow:", tf.__version__)

## 2. Загрузка данных

Fashion MNIST содержит 60 000 обучающих и 10 000 тестовых изображений. Каждое изображение — это матрица 28 × 28, а метка класса — число от 0 до 9.

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()

print("Обучающие изображения:", x_train.shape)
print("Обучающие метки:", y_train.shape)
print("Тестовые изображения:", x_test.shape)
print("Тестовые метки:", y_test.shape)

In [ ]:
class_names = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

print("Первые 10 меток:", y_train[:10])
print("Класс первого изображения:", class_names[y_train[0]])

## 3. Просмотр изображений

Посмотрим на первые 25 изображений и соответствующие им названия классов.

In [ ]:
fig, axes = plt.subplots(5, 5, figsize=(10, 10))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train[i], cmap="gray")
    ax.set_title(class_names[y_train[i]])
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Подготовка данных

Исходные значения пикселей находятся в диапазоне от 0 до 255. Делим их на 255, чтобы привести значения к диапазону от 0 до 1. Метки классов оставляем целыми числами — для них подходит функция потерь `sparse_categorical_crossentropy`.

In [ ]:
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

print("Диапазон пикселей:", float(x_train.min()), "—", float(x_train.max()))

## 5. Создание нейронной сети

Слой `Flatten` преобразует изображение 28 × 28 в вектор из 784 чисел. Затем идут два полносвязных скрытых слоя с активацией ReLU. Выходной слой содержит 10 нейронов — по одному на каждый класс; `softmax` превращает выходы в вероятности.

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(28, 28)),
    keras.layers.Flatten(name="flatten"),
    keras.layers.Dense(256, activation="relu", name="hidden_1"),
    keras.layers.Dense(128, activation="relu", name="hidden_2"),
    keras.layers.Dense(10, activation="softmax", name="output")
])

model.summary()

## 6. Настройка и обучение

Оптимизатор Adam обновляет веса сети, а точность (`accuracy`) показывает долю правильно классифицированных изображений. Используем 20% обучающих данных для проверки качества на каждой эпохе. `EarlyStopping` остановит обучение, если качество на валидации перестанет улучшаться, и восстановит лучшие веса.

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    x_train,
    y_train,
    epochs=20,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=1
)

## 7. Графики обучения

Сравним ошибку и точность на обучающей и валидационной выборках. Если обучающая ошибка продолжает уменьшаться, а валидационная растёт, это может указывать на переобучение.

In [ ]:
epochs_range = range(1, len(history.history["loss"]) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs_range, history.history["loss"], label="Обучение")
plt.plot(epochs_range, history.history["val_loss"], label="Валидация")
plt.xlabel("Эпоха")
plt.ylabel("Loss")
plt.title("Функция потерь")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(epochs_range, history.history["accuracy"], label="Обучение")
plt.plot(epochs_range, history.history["val_accuracy"], label="Валидация")
plt.xlabel("Эпоха")
plt.ylabel("Accuracy")
plt.title("Точность классификации")
plt.legend()
plt.grid(True)
plt.show()

## 8. Оценка на тестовой выборке

Тестовые данные не использовались при обучении и валидации модели. Поэтому оценка на них позволяет проверить, как сеть работает на ранее не виденных изображениях.

In [ ]:
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f} ({test_accuracy * 100:.2f}%)")

## 9. Примеры предсказаний

Для нескольких тестовых изображений выведем правильный класс, предсказанный класс и уверенность модели.

In [ ]:
probabilities = model.predict(x_test[:10], verbose=0)
predicted_classes = np.argmax(probabilities, axis=1)

fig, axes = plt.subplots(2, 5, figsize=(13, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_test[i], cmap="gray")
    correct = predicted_classes[i] == y_test[i]
    color = "green" if correct else "red"
    ax.set_title(
        f"Предсказание: {class_names[predicted_classes[i]]}\n"
        f"Верно: {class_names[y_test[i]]}\n"
        f"Уверенность: {probabilities[i].max():.1%}",
        color=color, fontsize=8
    )
    ax.axis("off")
plt.tight_layout()
plt.show()

## 10. Вывод

В этой работе была создана и обучена нейронная сеть для классификации изображений Fashion MNIST. Данные нормализованы, качество обучения проверено на валидационной выборке, а итоговая точность измерена на отдельной тестовой выборке. По графикам можно оценить динамику обучения и возможное переобучение.

**Перед отправкой:** запустите все ячейки сверху вниз и убедитесь, что обучение завершилось без ошибок. Затем в терминале CS50 выполните проверку задания:
```bash
npx cs204 2026/fashion local
```
Если проверка пройдёт, отправьте работу:
```bash
npx cs204 2026/fashion
```